# Basic DBS-Gym controller comparison

This notebook trains a small MLP-PPO policy and compares it with no DBS and constant high-frequency DBS (cDBS). It is a fast smoke test, not a scientific benchmark.

The default test uses `env0`, short episodes, one seed, and 256 PPO timesteps. Increase `PPO_STEPS` and `EPISODES` for a more meaningful experiment.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "environment").is_dir() and (REPO_ROOT.parent / "environment").is_dir():
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / "environment").is_dir():
    raise FileNotFoundError(f"Could not find repository root from {Path.cwd()}")

sys.path.insert(0, str(REPO_ROOT))
print(f"Repository root: {REPO_ROOT}")


Repository root: /Users/jesusvizcarra/Documents/GitHub/DBS-Gym


In [2]:
from copy import deepcopy

import numpy as np
from stable_baselines3 import PPO

from environment.env import SpatialKuramoto
from environment.env_configs.env0 import (
    coord_modif,
    grid_size,
    locus_center,
    locus_size,
    n_neurons,
    params_dict_train,
)
from environment.utils import calc_beta_band_power, generate_w0_with_locus, units2sec


SEED = 228
PPO_STEPS = 256
EPISODES = 2



## Environment and test settings

The PPO action is normalized to `[-1, 1]`. The environment rescales it to physical DBS amplitude `[-5, 5]`. Therefore no DBS is action `0.0`, while HF-DBS/cDBS uses action `1.0`, corresponding to physical amplitude `+5`.


In [3]:
def make_params():
    params = deepcopy(params_dict_train)
    params.update(
        reward_func="bbpow_action",
        # Short settings for a quick smoke test.
        transient_state_len=20.0,
        observe_wind_counts=20,
        total_episode_len=90.0,
        verbose=0,
        rand_seed=SEED,
    )

    w0, neur_coords, neur_grid, w0_without_locus, w_locus, locus_mask = (
        generate_w0_with_locus(
            n_neurons,
            grid_size,
            coord_modif,
            locus_center=locus_center,
            locus_size=locus_size,
            wmuL=params["wmuL"],
            wsdL=params["wsdL"],
            show=False,
        )
    )
    params.update(
        w0=w0,
        w0_without_locus=w0_without_locus,
        locus_without_w0=w_locus,
        locus_mask=locus_mask,
        neur_coords=neur_coords,
        neur_grid=neur_grid,
    )
    return params


def make_env(params):
    return SpatialKuramoto(params_dict=deepcopy(params))




## Evaluation functions

Each evaluation episode resets the simulator, runs until the episode terminates, and records reward, stimulation energy, and beta-band power.


In [4]:
def evaluate_constant(env, normalized_action, episodes):
    rewards = []
    energies = []
    beta_powers = []
    dt_seconds = units2sec(env.params_dict["verbose_dt"])

    for episode in range(episodes):
        # reset() uses NumPy's global RNG, so reseed to pair episodes across
        # controllers as closely as possible.
        np.random.seed(SEED + episode)
        env.reset()
        episode_reward = 0.0
        episode_energy = 0.0
        lfp = []
        terminated = False

        while not terminated:
            _, reward, terminated, truncated, _ = env.step(
                np.asarray([normalized_action], dtype=np.float32)
            )
            episode_reward += float(reward)
            episode_energy += float(np.abs(env.u).sum())
            lfp.append(env.theta_mean)
            if truncated:
                break

        signal = np.concatenate(lfp)
        beta_power = calc_beta_band_power(signal, dt_seconds, 12.5, 21.0)
        rewards.append(episode_reward)
        energies.append(episode_energy)
        beta_powers.append(beta_power)

    return {
        "reward_mean": float(np.mean(rewards)),
        "reward_sd": float(np.std(rewards, ddof=1)) if episodes > 1 else 0.0,
        "energy_mean": float(np.mean(energies)),
        "beta_power_mean": float(np.mean(beta_powers)),
    }


def evaluate_policy(model, env, episodes):
    rewards = []
    energies = []
    beta_powers = []
    dt_seconds = units2sec(env.params_dict["verbose_dt"])

    for episode in range(episodes):
        np.random.seed(SEED + episode)
        observation, _ = env.reset()
        episode_reward = 0.0
        episode_energy = 0.0
        lfp = []
        terminated = False

        while not terminated:
            action, _ = model.predict(observation, deterministic=True)
            observation, reward, terminated, truncated, _ = env.step(action)
            episode_reward += float(reward)
            episode_energy += float(np.abs(env.u).sum())
            lfp.append(env.theta_mean)
            if truncated:
                break

        signal = np.concatenate(lfp)
        beta_power = calc_beta_band_power(signal, dt_seconds, 12.5, 21.0)
        rewards.append(episode_reward)
        energies.append(episode_energy)
        beta_powers.append(beta_power)

    return {
        "reward_mean": float(np.mean(rewards)),
        "reward_sd": float(np.std(rewards, ddof=1)) if episodes > 1 else 0.0,
        "energy_mean": float(np.mean(energies)),
        "beta_power_mean": float(np.mean(beta_powers)),
    }



## Train PPO and evaluate all controllers


In [5]:
def main():
    np.random.seed(SEED)
    params = make_params()

    train_env = make_env(params)
    model = PPO(
        "MlpPolicy",
        train_env,
        n_steps=64,
        batch_size=64,
        seed=SEED,
        verbose=1,
        policy_kwargs={"net_arch": [64, 64]},
    )
    model.learn(total_timesteps=PPO_STEPS, progress_bar=True) # bug, terminal output not rendered correctly in notebook, but works fine in terminal

    results = {
        "MLP-PPO": evaluate_policy(model, make_env(params), EPISODES),
        "No DBS": evaluate_constant(make_env(params), 0.0, EPISODES),
        "HF-DBS/cDBS": evaluate_constant(make_env(params), 1.0, EPISODES),
    }

    print("\nController comparison")
    print("controller,reward_mean,reward_sd,energy_mean,beta_power_mean")
    for name, metrics in results.items():
        print(
            f"{name},{metrics['reward_mean']:.6g},{metrics['reward_sd']:.6g},"
            f"{metrics['energy_mean']:.6g},{metrics['beta_power_mean']:.6g}"
        )


if __name__ == "__main__":
    main()



No temporal drift events!
DBS affects 512 neurons, min=0.307 & max=1.0
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
DBS affects 512 neurons, min=0.307 & max=1.0


Output()

---------------------------
| time/              |    |
|    fps             | 13 |
|    iterations      | 1  |
|    time_elapsed    | 4  |
|    total_timesteps | 64 |
---------------------------


DBS affects 512 neurons, min=0.307 & max=1.0

-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 100           |
|    ep_rew_mean          | -2.87e+03     |
| time/                   |               |
|    fps                  | 14            |
|    iterations           | 2             |
|    time_elapsed         | 9             |
|    total_timesteps      | 128           |
| train/                  |               |
|    approx_kl            | 3.1520613e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.42         |
|    explained_variance   | 0.000141      |
|    learning_rate        | 0.0003        |
|    loss                 | 5.55e+04      |
|    n_updates            | 10            |
|    policy_gradient_loss | -0.000436     |
|    std                  | 0.998         |
|    value_loss           | 1.11e+05      |
-------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 100          |
|    ep_rew_mean          | -2.87e+03    |
| time/                   |              |
|    fps                  | 16           |
|    iterations           | 3            |
|    time_elapsed         | 11           |
|    total_timesteps      | 192          |
| train/                  |              |
|    approx_kl            | 6.372109e-06 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | 0.000741     |
|    learning_rate        | 0.0003       |
|    loss                 | 5.61e+04     |
|    n_updates            | 20           |
|    policy_gradient_loss | -0.000377    |
|    std                  | 0.998        |
|    value_loss           | 1.12e+05     |
------------------------------------------


DBS affects 512 neurons, min=0.307 & max=1.0

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 100          |
|    ep_rew_mean          | -2.92e+03    |
| time/                   |              |
|    fps                  | 15           |
|    iterations           | 4            |
|    time_elapsed         | 16           |
|    total_timesteps      | 256          |
| train/                  |              |
|    approx_kl            | 2.618879e-05 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | -0.00387     |
|    learning_rate        | 0.0003       |
|    loss                 | 5.55e+04     |
|    n_updates            | 30           |
|    policy_gradient_loss | -0.00079     |
|    std                  | 1            |
|    value_loss           | 1.11e+05     |
------------------------------------------


No temporal drift events!
DBS affects 512 neurons, min=0.307 & max=1.0
DBS affects 512 neurons, min=0.307 & max=1.0
DBS affects 512 neurons, min=0.307 & max=1.0
No temporal drift events!
DBS affects 512 neurons, min=0.307 & max=1.0
DBS affects 512 neurons, min=0.307 & max=1.0
DBS affects 512 neurons, min=0.307 & max=1.0
No temporal drift events!
DBS affects 512 neurons, min=0.307 & max=1.0
DBS affects 512 neurons, min=0.307 & max=1.0
DBS affects 512 neurons, min=0.307 & max=1.0

Controller comparison
controller,reward_mean,reward_sd,energy_mean,beta_power_mean
MLP-PPO,-3373.74,69.4322,4.38645,0.00347517
No DBS,-3392.89,71.802,0,0.00349637
HF-DBS/cDBS,-836.102,0.599319,500,0.000286768
